# Automated labeling of time-frequency regions (Python port of the MathWorks example)

Port of [Automated Labeling of Time-Frequency Regions for AI-Based Spectrum Sensing Applications](https://www.mathworks.com/help/signal/ug/automated-labeling-of-time-frequency-regions-for-ai-based-spectrum-sensing-applications.html) to the data of `01_create_dataset.ipynb` (LTE, DMR, P25, TETRA plus distractors).

The idea: find regions of interest (ROIs) in a spectrogram without labels, describe each region by a few features, cluster the regions, and let a person name each **cluster** once instead of drawing every box.

Every MATLAB step is one Python function:

| MATLAB step | MATLAB | Python function here |
|---|---|---|
| 1. spectrogram | `stft`, `abs` | `compute_stft` (IQ) / `image_to_magnitude` (stored image) |
| 2. smoothing | `imdilate` | `smooth_spectrogram` (+ `average_spectrogram`, `suppress_dc_spike`: two additions for this data) |
| 2. threshold | `adaptthresh`, `imbinarize` | `adaptive_threshold` |
| 2. regions | bounding boxes of the mask | `regions_from_mask` |
| 2. alternative | Segment Anything Model | `localize_regions_sam` (optional) |
| 3. false-alarm filter | `helperExtractFeatures` (median magnitude) | `region_median_magnitude`, `filter_regions` |
| 4. evaluation | boxes to masks, confusion matrices | `boxes_to_mask`, `pixel_metrics`, `detection_metrics` |
| 5. manual correction | Signal Labeler | `expert_correction` (ground truth plays the expert) |
| 6. features | duration, spread, skewness, entropy ... | `extract_region_features`, `zscore` |
| 7. clustering | `kmedoids` (cityblock, replicates) | `kmedoids` |
| 8. cluster naming | `helperMapClusterLabels` | `map_cluster_labels` |

**Two methods, same modules, one comparison:**

1. **Method 1, IQ in:** the raw complex IQ of a scene goes through `compute_stft`, then the modules.
2. **Method 2, image in:** the stored spectrogram image (uint8 dB, as written by the generator) goes through `image_to_magnitude`, then the same modules.
3. `compare_methods` puts both on one figure.

Because the stored image is the 8-bit-quantised STFT of the same IQ, the two inputs are the same scene seen two ways. With the default settings the STFT of method 1 uses the same grid as the image, so the remaining difference is quantisation (0.31 dB steps) and the -20 dB floor. Change `STFT_NFFT` / `STFT_HOP` to give the IQ route a different resolution (everything is evaluated in Hz and seconds, so the grids need not match).

In [1]:
import sys, json, time, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == "autolabeling" else str(Path.cwd()))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from numpy.lib.stride_tricks import sliding_window_view
from scipy.signal import get_window
from scipy.ndimage import grey_dilation, uniform_filter, binary_closing, label as ndi_label, find_objects
from scipy.spatial.distance import cdist
from scipy.stats import entropy as sp_entropy

from spectrum_common import (SetReader, FS, N_FFT, HOP, N_FRAMES, N_SAMPLES, DB_MIN, DB_MAX, FAMILIES, SNR_EDGES, SNR_LABELS)

## 1. Data and settings

The auto-labeling needs the **raw IQ** for method 1, and the default generator profiles do not store it. The helper below builds a small `smoke` copy **with IQ** by running `01_create_dataset.ipynb` headless with `SPECTRUM_STORE_IQ=1` into `dataset/with_iq/` (your other datasets are not touched, and `find_dataset_root` in `spectrum_common.py` does not see this folder). It does nothing if the data is already there.

In [2]:
import os, subprocess, tempfile, glob
PROJECT = Path(sys.path[0]).resolve() if (Path(sys.path[0]) / "01_create_dataset.ipynb").exists() else Path.cwd().parent
SET_NAME = "train"            # which set to auto-label (the stored sets are split by session; the labels are only used for evaluation)
N_SCENES = None               # None = all scenes of the set
K_CLUSTERS = 8                # 5 classes (LTE, DMR, P25, TETRA, UNKNOWN) + 3 spare clusters, as the MATLAB example over-clusters on purpose
FEATURES = ("duration", "spread", "skewness", "entropy")   # the MATLAB selection; try adding "bandwidth" and "central_freq"
IOU_THR = 0.5                 # box match threshold for precision / recall
RUN_SAM = False               # True: also try the SAM localiser (slow, see section 6)
STFT_NFFT, STFT_HOP = N_FFT, HOP        # STFT of the IQ route (method 1). Same grid as the stored images by default.
PARAMS = dict(dc_guard=2,               # bins around 0 Hz replaced by the noise floor (receiver DC spike)
              average=(15, 3),          # (frames, bins) power averaging, extension to the MathWorks chain
              dilate=(3, 1),            # imdilate structuring element (frames, bins)
              neighborhood=None,        # adaptthresh window: None = MATLAB default, only used with statistic="mean"
              statistic="row_median",   # "mean" = MATLAB adaptthresh, "row_median" = per-frame noise floor
              sensitivity=0.3,          # adaptthresh sensitivity (threshold = 1.4 x the noise floor)
              close=(13, 5),            # binary closing (frames, bins): merges the bursts of one transmission (<= 5 ms apart)
              min_size=(16, 2),         # smallest region (frames, bins)
              min_median_mag=1.0)       # keep regions whose median magnitude is above this (noise floor = about 0.9)


def ensure_iq_dataset(set_name=SET_NAME):
    """Return the root of a dataset that stores IQ; build the smoke profile with IQ if there is none yet."""
    found = sorted(glob.glob(str(PROJECT / "dataset" / "with_iq" / "spectrum_rf_v*_smoke")))
    if found and (Path(found[-1]) / "sets" / set_name / "_SUCCESS").exists():
        return Path(found[-1])
    env = dict(os.environ, SPECTRUM_PROFILE="smoke", SPECTRUM_STORE_IQ="1", SPECTRUM_OUT=str(PROJECT / "dataset" / "with_iq"), MPLBACKEND="Agg")
    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", str(PROJECT / "01_create_dataset.ipynb"),
                        "--output-dir", tmp, "--output", "gen.ipynb", "--ExecutePreprocessor.timeout=3600"], check=True, env=env, cwd=PROJECT)
    return Path(sorted(glob.glob(str(PROJECT / "dataset" / "with_iq" / "spectrum_rf_v*_smoke")))[-1])


ROOT = ensure_iq_dataset()
reader = SetReader(ROOT, SET_NAME, max_scenes=N_SCENES)
SCENES = list(range(len(reader)))
print(ROOT.name, "| set", SET_NAME, "|", len(reader), "scenes |", len(reader.signals), "labelled boxes | IQ stored:", reader.iq(0) is not None)

spectrum_rf_v0.4.0_smoke | set train | 160 scenes | 383 labelled boxes | IQ stored: True


## 2. Modules of the MATLAB workflow, one function each

### 2.1 Spectrogram (step 1)
`compute_stft` is the IQ route (method 1). `image_to_magnitude` is the image route (method 2): it undoes the dB quantisation so that both routes give a **linear magnitude** on which the same modules run. Unit-variance noise has a magnitude of about 0.9 in both.

In [3]:
def compute_stft(iq, fs=FS, n_fft=N_FFT, hop=HOP, n_frames=None):
    """MATLAB `stft` + abs: complex IQ -> linear magnitude [frames, bins] (bins ascending, -fs/2..+fs/2)
    and its time / frequency axes. Power is normalised by the window energy, so unit-variance complex noise
    has E[|X|^2] = 1 (the same reference as the dataset generator)."""
    iq = np.asarray(iq)
    win = get_window("hann", n_fft).astype(np.float32)
    fr = sliding_window_view(iq, n_fft)[::hop]
    if n_frames is not None:
        fr = fr[:n_frames]
    X = np.fft.fftshift(np.fft.fft(fr * win, axis=-1), axes=-1)
    mag = (np.abs(X) / np.sqrt(np.sum(win.astype(np.float64) ** 2))).astype(np.float32)
    t = (np.arange(len(fr)) * hop + n_fft / 2) / fs                      # centre of every window (s)
    f = (np.arange(n_fft) - n_fft / 2) * fs / n_fft                       # Hz
    return mag, dict(t=t, f=f, fs=fs, n_fft=n_fft, hop=hop)


def image_to_magnitude(spec_u8, db_min=DB_MIN, db_max=DB_MAX):
    """Spectrogram *image* (uint8 dB, as stored by 01_create_dataset.ipynb) -> linear magnitude.
    Same grid as the dataset (N_FFT bins, HOP); the 8-bit quantisation and the [db_min, db_max] clipping stay in."""
    db = np.asarray(spec_u8).astype(np.float32) * (db_max - db_min) / 255.0 + db_min
    mag = 10.0 ** (db / 20.0)
    t = (np.arange(mag.shape[0]) * HOP + N_FFT / 2) / FS
    f = (np.arange(mag.shape[1]) - mag.shape[1] / 2) * FS / mag.shape[1]
    return mag.astype(np.float32), dict(t=t, f=f, fs=FS, n_fft=mag.shape[1], hop=HOP)

### 2.2 Geometry
Everything is compared in physical units (Hz, seconds), so boxes from different STFT grids can be matched.

In [4]:
def pixel_to_physical(x0, y0, x1, y1, ax):
    """Inclusive pixel box (bins, frames) -> (f0 Hz, f1 Hz, t0 s, t1 s). Same convention as the dataset generator."""
    df, n, hop, fs = ax["fs"] / ax["n_fft"], ax["n_fft"], ax["hop"], ax["fs"]
    return ((x0 - n / 2 - 0.5) * df, (x1 - n / 2 + 0.5) * df,
            (y0 * hop + n / 2 - hop / 2) / fs, (y1 * hop + n / 2 + hop / 2) / fs)


def physical_to_pixel(f0, f1, t0, t1, ax, shape):
    """Inverse of pixel_to_physical, clipped to the image `shape` = (frames, bins)."""
    df, n, hop, fs = ax["fs"] / ax["n_fft"], ax["n_fft"], ax["hop"], ax["fs"]
    x0, x1 = int(round(f0 / df + n / 2 + 0.5)), int(round(f1 / df + n / 2 - 0.5))
    y0, y1 = int(round((t0 * fs - n / 2 + hop / 2) / hop)), int(round((t1 * fs - n / 2 - hop / 2) / hop))
    return max(x0, 0), max(y0, 0), min(x1, shape[1] - 1), min(y1, shape[0] - 1)


def iou_phys(a, b):
    """IoU matrix of two lists of (f0, f1, t0, t1) boxes in physical units."""
    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    iw = np.clip(np.minimum(a[:, None, 1], b[None, :, 1]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
    ih = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 2], b[None, :, 2]), 0, None)
    inter = iw * ih
    area = lambda z: (z[:, 1] - z[:, 0]) * (z[:, 3] - z[:, 2])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-12)

### 2.3 ROI localisation and filtering (steps 2 and 3)
`smooth_spectrogram` (`imdilate`), `adaptive_threshold` (`adaptthresh` + `imbinarize`), `regions_from_mask` (boxes), and the false-alarm filter on the median magnitude of each region. Two things differ from the MATLAB example because the signals here are narrower (a DMR / P25 channel is about 6 bins wide) and share the band with a 1.1 MHz LTE block:

- `average_spectrogram` averages the power over a few frames before thresholding, so a long narrowband burst stands out of the speckle (a plain MATLAB-style threshold gave a precision of a few percent on this data).
- `adaptive_threshold(statistic="row_median")` uses the per-frame noise floor instead of a local mean: inside a 1.1 MHz LTE block the local mean *is* the LTE level, so the block would never exceed it (recall 0 for LTE). `statistic="mean"` is the MATLAB behaviour.
- `suppress_dc_spike` removes the receiver's DC spike, which would otherwise be a false alarm in every scene.

In [5]:
def suppress_dc_spike(mag, guard_bins=2):
    """Receiver artefact: the centre bin (0 Hz) carries the DC offset of the ADC and would be detected as a narrow signal in
    every scene. The +-guard_bins around it are replaced by the per-frame median (the noise floor). 0 switches it off."""
    if guard_bins <= 0:
        return mag
    mag = mag.copy()
    c = mag.shape[1] // 2
    mag[:, c - guard_bins:c + guard_bins + 1] = np.median(mag, axis=1, keepdims=True)
    return mag


def smooth_spectrogram(mag, size=(3, 3)):
    """MATLAB `imdilate` (grey-scale, rectangular structuring element): closes small gaps in a signal's ridge."""
    return grey_dilation(mag, size=size)


def average_spectrogram(mag, size=(15, 5)):
    """Extension to the MathWorks chain: power averaging over a (frames, bins) window. A narrowband burst lasts tens of frames,
    noise does not, so the average lifts a 6-bin DMR / P25 ridge out of the speckle. size=(1, 1) switches it off."""
    if tuple(size) == (1, 1):
        return mag
    return np.sqrt(uniform_filter(mag.astype(np.float32) ** 2, size=size, mode="reflect"))


def adaptive_threshold(img, neighborhood=None, sensitivity=0.5, statistic="mean"):
    """MATLAB `adaptthresh` + `imbinarize` on a bright foreground: pixel > local statistic * gain, gain = 2*(1 - sensitivity)
    (sensitivity 0.5 = the plain statistic, higher = lower threshold = more foreground; same behaviour as MATLAB, not bit-identical).
    statistic = "mean"       : local mean over 2*floor(size/16)+1 pixels per dimension (the MATLAB default window).
    statistic = "row_median" : median over frequency of every frame, i.e. a noise-floor estimate per frame. Needed here because a
                               local mean inside a 1.1 MHz LTE block is the LTE level itself, so the block would never exceed it."""
    gain = 2.0 * (1.0 - sensitivity)
    if statistic == "mean":
        if neighborhood is None:
            neighborhood = tuple(2 * (s // 16) + 1 for s in img.shape)
        ref = uniform_filter(img, size=neighborhood, mode="reflect")
    elif statistic == "row_median":
        ref = np.median(img, axis=1, keepdims=True)
    else:
        raise ValueError(statistic)
    return img > ref * gain


def regions_from_mask(binary, ax, close_size=(1, 1), min_size=(1, 1)):
    """Connected components of a binary mask -> table of ROIs (pixel box and time-frequency box).
    `close_size` = (frames, bins) binary closing that merges fragments of one transmission."""
    if close_size != (1, 1):
        binary = binary_closing(binary, structure=np.ones(close_size, bool))
    lab, n = ndi_label(binary, structure=np.ones((3, 3), bool))
    rows = []
    for k, sl in enumerate(find_objects(lab), start=1):
        y0, y1, x0, x1 = sl[0].start, sl[0].stop - 1, sl[1].start, sl[1].stop - 1
        if y1 - y0 + 1 < min_size[0] or x1 - x0 + 1 < min_size[1]:
            continue
        f0, f1, t0, t1 = pixel_to_physical(x0, y0, x1, y1, ax)
        rows.append(dict(x0=x0, y0=y0, x1=x1, y1=y1, f0=f0, f1=f1, t0=t0, t1=t1))
    return pd.DataFrame(rows, columns=["x0", "y0", "x1", "y1", "f0", "f1", "t0", "t1"])


def region_median_magnitude(mag, regions):
    """MATLAB helperExtractFeatures: median magnitude inside every region (used to drop false alarms)."""
    return np.array([np.median(mag[r.y0:r.y1 + 1, r.x0:r.x1 + 1]) for r in regions.itertuples()])


def filter_regions(mag, regions, min_median_mag):
    """Drop regions whose median magnitude is below the threshold (noise blobs)."""
    regions = regions.copy()
    regions["median_mag"] = region_median_magnitude(mag, regions) if len(regions) else []
    return regions[regions.median_mag >= min_median_mag].reset_index(drop=True)


def localize_regions_threshold(mag, ax, p):
    """Localisation module 1: DC guard -> averaging -> imdilate -> adaptive threshold -> connected regions -> median-magnitude filter."""
    mag = suppress_dc_spike(mag, p["dc_guard"])
    sm = smooth_spectrogram(average_spectrogram(mag, p["average"]), p["dilate"])
    binary = adaptive_threshold(sm, p["neighborhood"], p["sensitivity"], p["statistic"])
    regions = regions_from_mask(binary, ax, p["close"], p["min_size"])
    return filter_regions(mag, regions, p["min_median_mag"])

### 2.4 Features and clustering (steps 6 to 8)
Region features, z-score, `kmedoids` with the cityblock distance and several replicates, and the cluster naming that stands in for the expert.

In [6]:
FEATURE_NAMES = ["duration", "bandwidth", "central_freq", "spread", "skewness", "kurtosis", "entropy"]


def extract_region_features(mag, regions, ax):
    """MATLAB feature step. For every region: duration, bandwidth, central frequency and the spectral spread, skewness,
    kurtosis and entropy of its time-averaged power spectrum (treated as a distribution over frequency)."""
    df, hop, fs = ax["fs"] / ax["n_fft"], ax["hop"], ax["fs"]
    rows = []
    for r in regions.itertuples():
        sub = mag[r.y0:r.y1 + 1, r.x0:r.x1 + 1].astype(np.float64) ** 2
        psd = sub.mean(0)
        p = psd / psd.sum()
        f = ax["f"][r.x0:r.x1 + 1]
        c = float(np.sum(p * f))
        spread = float(np.sqrt(np.sum(p * (f - c) ** 2)))
        skew = float(np.sum(p * (f - c) ** 3) / spread ** 3) if spread > 0 else 0.0
        kurt = float(np.sum(p * (f - c) ** 4) / spread ** 4) if spread > 0 else 0.0
        ent = float(sp_entropy(p, base=2) / np.log2(len(p))) if len(p) > 1 else 0.0
        rows.append(dict(duration=(r.y1 - r.y0 + 1) * hop / fs, bandwidth=(r.x1 - r.x0 + 1) * df,
                         central_freq=c, spread=spread, skewness=skew, kurtosis=kurt, entropy=ent))
    return pd.DataFrame(rows, columns=FEATURE_NAMES)


def zscore(X):
    """MATLAB `normalize` (z-score per column); returns the scaled matrix and the (mean, std) used."""
    mu, sd = X.mean(0), X.std(0)
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mu) / sd, (mu, sd)

In [7]:
def kmedoids(X, k, metric="cityblock", replicates=50, max_iter=100, seed=0):
    """MATLAB `kmedoids(X, k, Distance="cityblock", Replicates=...)` (alternating PAM with k-means++ style seeding).
    Returns labels [n], medoid indices [k] and the total within-cluster distance of the best replicate."""
    rng = np.random.default_rng(seed)
    D = cdist(X, X, metric=metric)
    n, k = len(X), min(k, len(X))
    best = (np.inf, None, None)
    for _ in range(replicates):
        med = [int(rng.integers(n))]
        while len(med) < k:                                   # k-means++: next medoid far from the chosen ones
            d = D[:, med].min(1) ** 2
            med.append(int(rng.choice(n, p=d / d.sum())) if d.sum() > 0 else int(rng.integers(n)))
        med = np.array(med)
        for _ in range(max_iter):
            lab = np.argmin(D[:, med], 1)
            new = med.copy()
            for j in range(k):
                members = np.flatnonzero(lab == j)
                if len(members):
                    new[j] = members[np.argmin(D[np.ix_(members, members)].sum(1))]
            if np.array_equal(new, med):
                break
            med = new
        lab = np.argmin(D[:, med], 1)
        cost = D[np.arange(n), med[lab]].sum()
        if cost < best[0]:
            best = (cost, lab, med)
    return best[1], best[2], float(best[0])


def map_cluster_labels(clusters, true_labels):
    """MATLAB helperMapClusterLabels: name every cluster after its most frequent true class (this stands in for the
    expert who would look at a few members of each cluster and type one name for the whole batch)."""
    mapping = {}
    for c in np.unique(clusters):
        vals, counts = np.unique(np.asarray(true_labels)[clusters == c], return_counts=True)
        mapping[int(c)] = vals[np.argmax(counts)]
    return mapping

### 2.5 One scene end to end, and the evaluation (steps 3 to 5)
`detect_scene` runs steps 1 to 3 for one scene with either input. `expert_correction` keeps the regions that overlap a ground-truth box (IoU >= 0.3) and gives them the true family, as the expert would. The metrics compare boxes (precision, recall) and pixels (confusion matrix against the stored mask).

In [8]:
def get_scene_input(reader, i, source):
    """Input of the two methods for scene i.
    source = "iq"    : raw IQ -> compute_stft            (method 1)
    source = "image" : stored uint8 dB spectrogram image  (method 2)
    Both return (linear magnitude, axes dict)."""
    if source == "iq":
        iq = reader.iq(i)
        if iq is None:
            raise FileNotFoundError("this set has no IQ: build it with SPECTRUM_STORE_IQ=1 (see the first cells)")
        return compute_stft(iq, FS, STFT_NFFT, STFT_HOP)
    if source == "image":
        return image_to_magnitude(reader.spec_u8(i))
    raise ValueError(source)


def gt_boxes_physical(signals, ax):
    """Ground-truth boxes of one scene as (f0, f1, t0, t1) in Hz / s, plus their family labels and SNR."""
    if len(signals) == 0:
        return np.zeros((0, 4)), [], np.zeros(0)
    ax_ds = dict(fs=FS, n_fft=N_FFT, hop=HOP)              # the dataset grid: signals.box_* are pixels of it
    boxes = np.array([pixel_to_physical(r.box_x0, r.box_y0, r.box_x1, r.box_y1, ax_ds) for r in signals.itertuples()])
    return boxes, list(signals.family), signals.snr_db.to_numpy()


_MASK_CACHE = {}
def load_gt_mask(reader, i):
    """Pixel ground truth (family id per pixel, 0 = background) of scene i from the set's mask shard."""
    r = reader.scenes.iloc[i]
    key = (str(reader.dir), int(r["shard"]))
    if key not in _MASK_CACHE:
        _MASK_CACHE.clear()
        _MASK_CACHE[key] = np.load(reader.dir / f"shard_{int(r['shard']):04d}_mask.npz")["mask"]
    return _MASK_CACHE[key][int(r["pos"])]


def boxes_to_mask(regions, shape):
    """Regions -> binary mask (MATLAB: convert bounding boxes to masks before the pixel comparison)."""
    m = np.zeros(shape, bool)
    for r in regions.itertuples():
        m[r.y0:r.y1 + 1, r.x0:r.x1 + 1] = True
    return m

In [9]:
def detect_scene(reader, i, source, params, localizer="threshold", sam_model=None):
    """Steps 1 to 3 of the MATLAB example for one scene: spectrogram (or image) -> ROI localisation -> filtering.
    Returns the regions table, the ground truth and the pixel masks."""
    mag, ax = get_scene_input(reader, i, source)
    if localizer == "threshold":
        regions = localize_regions_threshold(mag, ax, params)
    elif localizer == "sam":
        regions = localize_regions_sam(mag, ax, params, sam_model)
    else:
        raise ValueError(localizer)
    gt_box, gt_lab, gt_snr = gt_boxes_physical(reader.boxes(i), ax)
    gt_mask = load_gt_mask(reader, i) > 0
    if gt_mask.shape != mag.shape:                                      # IQ route with another STFT grid
        gt_mask = cv2.resize(gt_mask.astype(np.uint8), (mag.shape[1], mag.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
    regions = regions.copy()
    feats = extract_region_features(mag, regions, ax)
    regions = pd.concat([regions.reset_index(drop=True), feats], axis=1)
    regions["scene"] = i
    return dict(scene=i, regions=regions, gt_box=gt_box, gt_label=gt_lab, gt_snr=gt_snr,
                pred_mask=boxes_to_mask(regions, mag.shape), gt_mask=gt_mask)


def expert_correction(res, iou_thr=0.3):
    """MATLAB step 'manual correction'. The expert deletes false alarms and keeps the true regions. Here ground truth plays the
    expert: a region is kept (and gets the true family) when it overlaps a ground-truth box with IoU >= iou_thr.
    Adds columns `true_label`, `gt_index`, `iou` to res['regions'] (-1 / '' for false alarms)."""
    reg = res["regions"]
    m, best = match_boxes(reg[["f0", "f1", "t0", "t1"]].values, res["gt_box"], iou_thr)
    reg["gt_index"], reg["iou"] = m, best
    reg["true_label"] = [res["gt_label"][g] if g >= 0 else "" for g in m]
    return res


def run_method(reader, idxs, source, params, localizer="threshold", sam_model=None, verbose=True):
    """Run detect_scene on many scenes with one input type. Returns the list of per-scene results."""
    out = []
    t0 = time.time()
    for i in idxs:
        out.append(expert_correction(detect_scene(reader, i, source, params, localizer, sam_model)))
    if verbose:
        print(f"{source:5s} / {localizer:9s}: {len(out)} scenes, {sum(len(r['regions']) for r in out)} regions, {time.time() - t0:.1f} s")
    return out

In [10]:
def match_boxes(pred, gt, thr=0.5):
    """Greedy one-to-one matching on IoU (largest first). Returns (matched gt index per prediction or -1, iou per prediction)."""
    pred, gt = np.asarray(pred, float).reshape(-1, 4), np.asarray(gt, float).reshape(-1, 4)
    out, best = np.full(len(pred), -1), np.zeros(len(pred))
    if len(pred) == 0 or len(gt) == 0:
        return out, best
    iou = iou_phys(pred, gt)
    work = iou.copy()
    while True:
        p, g = np.unravel_index(np.argmax(work), work.shape)
        if work[p, g] < thr:
            break
        out[p], best[p] = g, work[p, g]
        work[p, :], work[:, g] = -1, -1
    return out, best

In [11]:
def detection_metrics(results, iou_thr=0.5):
    """Box-level detection quality (TP / FP / FN, precision, recall, F1, mean IoU of the matches) and the per-GT table
    (family, SNR, detected?) used for the recall-versus-SNR curve."""
    tp = fp = fn = 0
    ious, gt_rows = [], []
    for r in results:
        pred = r["regions"][["f0", "f1", "t0", "t1"]].values
        m, bi = match_boxes(pred, r["gt_box"], iou_thr)
        tp += int((m >= 0).sum()); fp += int((m < 0).sum()); fn += len(r["gt_box"]) - int((m >= 0).sum())
        ious += list(bi[m >= 0])
        hit = set(m[m >= 0].tolist())
        for g in range(len(r["gt_box"])):
            gt_rows.append(dict(scene=r["scene"], family=r["gt_label"][g], snr_db=r["gt_snr"][g], detected=g in hit))
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return dict(tp=tp, fp=fp, fn=fn, precision=prec, recall=rec, f1=2 * prec * rec / max(prec + rec, 1e-9),
                mean_iou=float(np.mean(ious)) if ious else 0.0), pd.DataFrame(gt_rows)


def pixel_metrics(results):
    """Pixel-level comparison of the union of the predicted boxes with the ground-truth mask (MATLAB confusion matrices)."""
    cm = np.zeros((2, 2), np.int64)                                     # rows: truth (bg, signal); cols: prediction
    for r in results:
        p, g = r["pred_mask"].ravel(), r["gt_mask"].ravel()
        cm[0, 0] += np.sum(~g & ~p); cm[0, 1] += np.sum(~g & p); cm[1, 0] += np.sum(g & ~p); cm[1, 1] += np.sum(g & p)
    tp, fp, fn = cm[1, 1], cm[0, 1], cm[1, 0]
    return dict(pixel_precision=tp / max(tp + fp, 1), pixel_recall=tp / max(tp + fn, 1), pixel_iou=tp / max(tp + fp + fn, 1),
                pixel_accuracy=(cm[0, 0] + cm[1, 1]) / cm.sum()), cm


def cluster_and_label(results, features=("duration", "spread", "skewness", "entropy"), k=8, replicates=50, seed=0):
    """MATLAB steps 'feature extraction -> k-medoids -> cluster labelling' on the expert-corrected regions of all scenes.
    Returns a table of the clustered regions with `cluster` and `pred_label`, plus the accuracy of the auto labels."""
    reg = pd.concat([r["regions"] for r in results], ignore_index=True)
    reg = reg[reg.true_label != ""].reset_index(drop=True)
    X, _ = zscore(reg[list(features)].to_numpy(float))
    cl, med, cost = kmedoids(X, k, "cityblock", replicates, seed=seed)
    mapping = map_cluster_labels(cl, reg.true_label.to_numpy())
    reg["cluster"], reg["pred_label"] = cl, [mapping[c] for c in cl]
    reg["correct"] = reg.pred_label == reg.true_label
    return reg, dict(label_accuracy=float(reg.correct.mean()), n_regions=len(reg), k=k, cost=cost)


def end_to_end(results, labelled, iou_thr=0.3):
    """Share of ground-truth boxes that ended up with the right label: detected AND correctly named."""
    n_gt = sum(len(r["gt_box"]) for r in results)
    ok = int(labelled.correct.sum()) if len(labelled) else 0
    return ok / max(n_gt, 1)

In [12]:
FAM_COLOR = {"LTE": "white", "DMR": "yellow", "P25": "cyan", "TETRA": "lime", "UNKNOWN": "orange"}
CLASS_ORDER = ["LTE", "DMR", "P25", "TETRA", "UNKNOWN"]


def show_spectrogram(ax_, mag, ax, title="", vmin=-5, vmax=30):
    """Magnitude -> dB image with frequency in kHz and time in ms."""
    db = 20 * np.log10(mag + 1e-6)
    ext = [ax["f"][0] / 1e3, ax["f"][-1] / 1e3, ax["t"][-1] * 1e3, ax["t"][0] * 1e3]
    ax_.imshow(db, aspect="auto", extent=ext, cmap="viridis", vmin=vmin, vmax=vmax)
    ax_.set_xlabel("frequency (kHz)"); ax_.set_ylabel("time (ms)"); ax_.set_title(title, fontsize=9)


def draw_boxes(ax_, boxes, color, ls="-", lw=1.2, labels=None):
    for k, (f0, f1, t0, t1) in enumerate(np.asarray(boxes).reshape(-1, 4)):
        ax_.add_patch(Rectangle((f0 / 1e3, t0 * 1e3), (f1 - f0) / 1e3, (t1 - t0) * 1e3, fill=False, ec=color, lw=lw, ls=ls))
        if labels is not None:
            ax_.text(f0 / 1e3, t0 * 1e3, labels[k], color=color, fontsize=6, va="bottom")


def plot_detections(ax_, mag, axes, res, title):
    """Spectrogram + ground truth (white dashed) + detections (green = matches a ground-truth box, red = false alarm)."""
    show_spectrogram(ax_, mag, axes, title)
    draw_boxes(ax_, res["gt_box"], "white", "--", 1.0)
    reg = res["regions"]
    ok = reg.gt_index.values >= 0
    draw_boxes(ax_, reg.loc[ok, ["f0", "f1", "t0", "t1"]].values, "lime", "-", 1.4)
    draw_boxes(ax_, reg.loc[~ok, ["f0", "f1", "t0", "t1"]].values, "red", "-", 1.4)


def pick_example(results, min_boxes=3):
    """A readable scene: several ground-truth boxes, the strongest median SNR among those."""
    cand = [r for r in results if len(r["gt_box"]) >= min_boxes] or results
    return max(cand, key=lambda r: np.median(r["gt_snr"]) if len(r["gt_snr"]) else -99)["scene"]


def confusion_counts(labelled, classes=CLASS_ORDER):
    cm = np.zeros((len(classes), len(classes)), int)
    for t, p in zip(labelled.true_label, labelled.pred_label):
        if t in classes and p in classes:
            cm[classes.index(t), classes.index(p)] += 1
    return cm


def plot_confusion(ax_, labelled, title, classes=CLASS_ORDER):
    cm = confusion_counts(labelled, classes)
    ax_.imshow(cm / np.maximum(cm.sum(1, keepdims=True), 1), cmap="Blues", vmin=0, vmax=1)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax_.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm[i].sum() * 0.5 else "black")
    ax_.set_xticks(range(len(classes)), classes, rotation=45, fontsize=7); ax_.set_yticks(range(len(classes)), classes, fontsize=7)
    ax_.set_xlabel("auto label"); ax_.set_ylabel("true"); ax_.set_title(title, fontsize=9)


def plot_pipeline_steps(reader, i, source, params):
    """The MATLAB walk-through for one scene: spectrogram, averaging, dilation, binary mask, regions, filtered regions."""
    mag, ax = get_scene_input(reader, i, source)
    avg = average_spectrogram(mag, params["average"])
    sm = smooth_spectrogram(avg, params["dilate"])
    binary = adaptive_threshold(sm, params["neighborhood"], params["sensitivity"], params["statistic"])
    raw = regions_from_mask(binary, ax, params["close"], params["min_size"])
    kept = filter_regions(mag, raw, params["min_median_mag"])
    fig, a = plt.subplots(2, 3, figsize=(16, 7))
    a = a.ravel()
    show_spectrogram(a[0], mag, ax, "1. STFT magnitude")
    show_spectrogram(a[1], avg, ax, "2. power averaging (extension)")
    show_spectrogram(a[2], sm, ax, "3. imdilate")
    a[3].imshow(binary, aspect="auto", cmap="gray"); a[3].set_title("4. adaptive threshold", fontsize=9)
    show_spectrogram(a[4], mag, ax, f"5. regions ({len(raw)})"); draw_boxes(a[4], raw[["f0", "f1", "t0", "t1"]].values, "red")
    show_spectrogram(a[5], mag, ax, f"6. median-magnitude filter ({len(kept)} kept)"); draw_boxes(a[5], kept[["f0", "f1", "t0", "t1"]].values, "lime")
    gb = gt_boxes_physical(reader.boxes(i), ax)[0]
    for k in (4, 5):
        draw_boxes(a[k], gb, "white", "--", 1.0)
    fig.suptitle(f"Scene {i}, input = {source}   (white dashed = ground truth)", fontsize=10)
    plt.tight_layout()
    return fig


def compare_methods(reader, res_iq, res_img, lab_iq, lab_img, info_iq, info_img, iou_thr=0.5, save=None):
    """ONE figure that compares method 1 (IQ -> STFT -> modules) with method 2 (spectrogram image -> modules)."""
    names = ["IQ -> STFT", "Spectrogram image"]
    res, labs, infos = (res_iq, res_img), (lab_iq, lab_img), (info_iq, info_img)
    det = [detection_metrics(r, iou_thr) for r in res]
    pix = [pixel_metrics(r)[0] for r in res]
    e2e = [end_to_end(r, l) for r, l in zip(res, labs)]
    colors = ["tab:blue", "tab:orange"]

    fig = plt.figure(figsize=(17, 13))
    gs = fig.add_gridspec(3, 3, height_ratios=[1.15, 1, 1], hspace=0.38, wspace=0.28)
    ex = pick_example(res_iq)
    scene_res = [next(r for r in rs if r["scene"] == ex) for rs in res]
    mags = [get_scene_input(reader, ex, s) for s in ("iq", "image")]
    # row 1: the same scene seen by both methods and their difference
    a0, a1, a2 = (fig.add_subplot(gs[0, k]) for k in range(3))
    for a, nm, (mag, axx), sr in zip((a0, a1), names, mags, scene_res):
        plot_detections(a, mag, axx, sr, f"{nm}, scene {ex}\n(green = hit, red = false alarm, white dashed = truth)")
    diff = 20 * np.log10(mags[0][0] + 1e-6) - 20 * np.log10(mags[1][0] + 1e-6)
    im = a2.imshow(diff, aspect="auto", cmap="coolwarm", vmin=-3, vmax=3,
                   extent=[mags[0][1]["f"][0] / 1e3, mags[0][1]["f"][-1] / 1e3, mags[0][1]["t"][-1] * 1e3, mags[0][1]["t"][0] * 1e3])
    a2.set_title("input difference: IQ-STFT minus image (dB)\n(8-bit quantisation and the -20 dB floor)", fontsize=9)
    a2.set_xlabel("frequency (kHz)"); a2.set_ylabel("time (ms)"); fig.colorbar(im, ax=a2, fraction=0.04)
    # row 2: headline metrics, recall versus SNR, recall per family
    b0, b1, b2 = (fig.add_subplot(gs[1, k]) for k in range(3))
    keys = [("precision", [d[0]["precision"] for d in det]), ("recall", [d[0]["recall"] for d in det]), ("F1", [d[0]["f1"] for d in det]),
            ("pixel IoU", [p["pixel_iou"] for p in pix]), ("label acc.\n(clusters)", [i["label_accuracy"] for i in infos]), ("end-to-end", e2e)]
    w = 0.38
    for k, (nm, c) in enumerate(zip(names, colors)):
        vals = [v[1][k] for v in keys]
        bars = b0.bar(np.arange(len(keys)) + (k - 0.5) * w, vals, w, label=nm, color=c)
        for x, v in zip(np.arange(len(keys)) + (k - 0.5) * w, vals):
            b0.text(x, v + 0.01, f"{v:.2f}", ha="center", fontsize=6)
    b0.set_xticks(range(len(keys)), [v[0] for v in keys], fontsize=7); b0.set_ylim(0, 1.1); b0.legend(fontsize=7)
    b0.set_title(f"metrics (IoU >= {iou_thr} for boxes)", fontsize=9)
    edges = np.array([-10, 5, 10, 15, 20, 25, 40]); mid = (edges[:-1] + edges[1:]) / 2
    for (d, g), nm, c in zip(det, names, colors):
        g = g.assign(b=pd.cut(g.snr_db, edges, labels=False))
        b1.plot(mid[:6][g.groupby("b").detected.mean().index.astype(int)], g.groupby("b").detected.mean().values, "-o", color=c, label=nm, ms=4)
    b1.set_xlabel("SNR over the occupied bandwidth (dB)"); b1.set_ylabel("recall"); b1.set_ylim(0, 1.05); b1.grid(alpha=0.3)
    b1.set_title("recall versus SNR", fontsize=9); b1.legend(fontsize=7)
    for k, ((d, g), nm, c) in enumerate(zip(det, names, colors)):
        rec = g.groupby("family").detected.mean().reindex(CLASS_ORDER)
        b2.bar(np.arange(len(CLASS_ORDER)) + (k - 0.5) * w, rec.values, w, color=c, label=nm)
    b2.set_xticks(range(len(CLASS_ORDER)), CLASS_ORDER, fontsize=8); b2.set_ylim(0, 1.05); b2.set_title("recall per protocol", fontsize=9)
    # row 3: auto-label confusion matrices and a summary table
    c0, c1, c2 = (fig.add_subplot(gs[2, k]) for k in range(3))
    plot_confusion(c0, lab_iq, f"{names[0]}: auto labels (accuracy {info_iq['label_accuracy']:.2f})")
    plot_confusion(c1, lab_img, f"{names[1]}: auto labels (accuracy {info_img['label_accuracy']:.2f})")
    c2.axis("off")
    rows = [["", names[0], names[1]],
            ["scenes", len(res_iq), len(res_img)],
            ["regions found", sum(len(r["regions"]) for r in res_iq), sum(len(r["regions"]) for r in res_img)],
            ["TP / FP / FN", *[f"{d[0]['tp']} / {d[0]['fp']} / {d[0]['fn']}" for d in det]],
            ["mean IoU of hits", *[f"{d[0]['mean_iou']:.3f}" for d in det]],
            ["pixel precision", *[f"{p['pixel_precision']:.3f}" for p in pix]],
            ["pixel recall", *[f"{p['pixel_recall']:.3f}" for p in pix]],
            ["clusters (k)", infos[0]["k"], infos[1]["k"]],
            ["label accuracy", *[f"{i['label_accuracy']:.3f}" for i in infos]],
            ["end-to-end", *[f"{v:.3f}" for v in e2e]]]
    tb = c2.table(cellText=rows, loc="center", cellLoc="center"); tb.auto_set_font_size(False); tb.set_fontsize(8); tb.scale(1, 1.35)
    fig.suptitle("Automated time-frequency labelling: method 1 (IQ -> STFT) versus method 2 (spectrogram image), same modules", fontsize=12, y=0.995)
    if save:
        fig.savefig(save, dpi=130, bbox_inches="tight")
    return fig

## 3. Look at the steps on one scene
The same walk-through as the MATLAB page: spectrogram, smoothing, binary mask, regions, filtered regions.

In [13]:
EX = pick_example(run_method(reader, SCENES[:40], "iq", PARAMS, verbose=False))
plot_pipeline_steps(reader, EX, "iq", PARAMS)
plt.show()

/var/folders/w8/ch1lylzj4jl9d20n1d12d57h0000gp/T/ipykernel_8143/1142332243.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Run both methods on the whole set
Method 1 starts from the IQ, method 2 from the stored image. Both then run the same detection, expert-correction, feature, clustering and naming functions. The clustering is done per method over the regions of all scenes.

In [14]:
RES, LAB, INFO = {}, {}, {}
for source in ("iq", "image"):
    RES[source] = run_method(reader, SCENES, source, PARAMS)
    LAB[source], INFO[source] = cluster_and_label(RES[source], FEATURES, K_CLUSTERS)

rows = []
for source, name in (("iq", "method 1: IQ -> STFT"), ("image", "method 2: spectrogram image")):
    d, _ = detection_metrics(RES[source], IOU_THR)
    pm, _ = pixel_metrics(RES[source])
    rows.append({"method": name, **{k: round(float(v), 3) for k, v in {**d, **pm}.items()},
                 "label_accuracy": round(INFO[source]["label_accuracy"], 3), "end_to_end": round(end_to_end(RES[source], LAB[source]), 3)})
pd.DataFrame(rows).set_index("method").T

iq    / threshold: 160 scenes, 466 regions, 9.3 s


image / threshold: 160 scenes, 468 regions, 8.6 s


method,method 1: IQ -> STFT,method 2: spectrogram image
tp,291.000,291.000
fp,175.000,177.000
fn,92.000,92.000
precision,0.624,0.622
recall,0.760,0.760
f1,0.686,0.684
mean_iou,0.732,0.731
pixel_precision,0.956,0.956
pixel_recall,0.930,0.930
pixel_iou,0.892,0.892


## 5. The comparison figure
Top row: the same scene as seen by both methods (green = region that matches a ground-truth box, red = false alarm, white dashed = ground truth) and the dB difference of the two inputs. Middle row: headline metrics, recall versus SNR, recall per protocol. Bottom row: confusion matrices of the automatic labels and a summary table.

In [15]:
fig = compare_methods(reader, RES["iq"], RES["image"], LAB["iq"], LAB["image"], INFO["iq"], INFO["image"], iou_thr=IOU_THR,
                      save=str(PROJECT / "autolabeling" / "comparison_iq_vs_image.png"))
plt.show()

/var/folders/w8/ch1lylzj4jl9d20n1d12d57h0000gp/T/ipykernel_8143/946831076.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Optional: SAM as the localiser
The MATLAB page also localises regions with the Segment Anything Model. `localize_regions_sam` does the same with MobileSAM (39 MB, downloaded on first use). It is slow on a CPU (several seconds per tile), so it runs on a few scenes only and only when `RUN_SAM = True`.

In [16]:
def load_sam(weights="mobile_sam.pt", device="cpu"):
    """Pre-trained Segment Anything Model (MobileSAM through ultralytics). Downloads the weights (39 MB) on first use."""
    from ultralytics import SAM
    return SAM(weights), device


def localize_regions_sam(mag, ax, p, sam, n_tiles=4):
    """Localisation module 2: SAM 'segment everything' on the smoothed spectrogram (MATLAB runs SAM on the imdilated image).
    The wide, short spectrogram is cut into `n_tiles` frequency tiles so that a 6-bin signal is not shrunk away by SAM's
    1024-pixel input; masks that fill most of a tile are the background and are dropped."""
    model, device = sam
    mag_c = suppress_dc_spike(mag, p["dc_guard"])
    sm = smooth_spectrogram(average_spectrogram(mag_c, p["average"]), p["dilate"])
    db = 20 * np.log10(sm + 1e-6)
    img = np.clip((db + 3) / 18 * 255, 0, 255).astype(np.uint8)
    w = img.shape[1] // n_tiles
    rows = []
    for k in range(n_tiles):
        tile = np.repeat(img[:, k * w:(k + 1) * w, None], 3, axis=2)
        res = model(tile, verbose=False, device=device)[0]
        if res.masks is None:
            continue
        for m in res.masks.data.cpu().numpy() > 0.5:
            if m.mean() > 0.5:
                continue
            ys, xs = np.where(m)
            x0, x1, y0, y1 = xs.min() + k * w, xs.max() + k * w, ys.min(), ys.max()
            f0, f1, t0, t1 = pixel_to_physical(x0, y0, x1, y1, ax)
            rows.append(dict(x0=x0, y0=y0, x1=x1, y1=y1, f0=f0, f1=f1, t0=t0, t1=t1))
    reg = pd.DataFrame(rows, columns=["x0", "y0", "x1", "y1", "f0", "f1", "t0", "t1"])
    if len(reg):
        reg = reg[((reg.y1 - reg.y0 + 1) >= p["min_size"][0]) & ((reg.x1 - reg.x0 + 1) >= p["min_size"][1])]
        iou = iou_phys(reg[["f0", "f1", "t0", "t1"]].values, reg[["f0", "f1", "t0", "t1"]].values)
        area = ((reg.f1 - reg.f0) * (reg.t1 - reg.t0)).values
        order, kept = np.argsort(-area), []
        for a in order:                                   # drop near-duplicate masks, biggest first
            if all(iou[a, b] <= 0.7 for b in kept):
                kept.append(a)
        reg = reg.iloc[sorted(kept)].reset_index(drop=True)
    return filter_regions(mag, reg, p["min_median_mag"])

In [17]:
if RUN_SAM:
    N_SAM = 16
    sam = load_sam(str(PROJECT / "autolabeling" / "mobile_sam.pt"))
    rows = []
    for source in ("iq", "image"):
        for loc in ("threshold", "sam"):
            res = run_method(reader, SCENES[:N_SAM], source, PARAMS, loc, sam)
            d, _ = detection_metrics(res, IOU_THR); pm, _ = pixel_metrics(res)
            rows.append(dict(input=source, localiser=loc, regions=sum(len(r["regions"]) for r in res), precision=d["precision"], recall=d["recall"], f1=d["f1"], pixel_iou=pm["pixel_iou"]))
    display(pd.DataFrame(rows).round(3))

## 7. Notes and limits
- **Expert step:** the expert is simulated with ground truth. With real recordings you would delete the false alarms and name each cluster by hand (Signal Labeler in MATLAB, Label Studio or CVAT here). The `label_accuracy` and `end_to_end` numbers are therefore the accuracy *if the expert names every cluster by its majority class*.
- **Clusters:** the confusion matrices show where the four features cannot separate protocols (DMR / P25 are both 4-level FSK of the same bandwidth). Add `bandwidth` and `central_freq` to `FEATURES`, or temporal features, to see the effect.
- **Low SNR:** narrowband signals at 5 to 10 dB over their occupied bandwidth are below the noise in each bin, so they are missed by any method that only sees one spectrogram. Recall versus SNR in the figure shows it.
- **Scale:** this runs on the `smoke` profile (a few hundred scenes). Larger profiles need no code change except `ensure_iq_dataset` (IQ is 0.13 MB per scene).